In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from db import engine

plt.rcParams["figure.figsize"] = (15, 6)
sns.set_style('whitegrid')

In [2]:
with engine.connect() as conn:
    df = pd.read_sql("select * from avg_daily_stats", conn)

In [3]:
df.head()

,city,state,observed_date,avg_temperature,avg_feels_like,avg_humidity,avg_pressure,avg_wind_speed
0,Albuquerque,NM,2023-01-01,6.05,3.20,67.92,840.69,5.39
1,Albuquerque,NM,2023-01-02,5.50,1.98,72.71,840.31,10.62
2,Albuquerque,NM,2023-01-03,2.64,-1.64,55.58,844.99,9.97
3,Albuquerque,NM,2023-01-04,2.03,-1.60,65.04,849.82,6.55
4,Albuquerque,NM,2023-01-05,1.48,-2.24,61.17,850.56,6.20


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32880 entries, 0 to 32879
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   city             32880 non-null  str    
 1   state            32880 non-null  str    
 2   observed_date    32880 non-null  object 
 3   avg_temperature  32880 non-null  float64
 4   avg_feels_like   32880 non-null  float64
 5   avg_humidity     32880 non-null  float64
 6   avg_pressure     32880 non-null  float64
 7   avg_wind_speed   32880 non-null  float64
dtypes: float64(5), object(1), str(2)
memory usage: 2.0+ MB


In [5]:
df['observed_date'] = pd.to_datetime(df['observed_date'])

In [6]:
df['year_month'] = df['observed_date'].dt.strftime("%Y-%m")

In [7]:
monthly_avg_temp_city = pd.pivot_table(data=df, columns='year_month', index='city', values='avg_temperature')

In [8]:
monthly_avg_temp_city

year_month,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,...,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
city,,,,,,,,,,,,,,,,,,,,,
Albuquerque,2.689355,3.920357,7.997742,13.786667,20.192903,24.667000,30.443226,26.917097,22.768333,16.086129,...,10.862581,15.373333,19.101935,26.226667,26.714839,27.212903,21.715667,16.697419,10.306333,6.835484
Anchorage,-5.425484,-6.679286,-4.269032,-0.328333,8.043871,12.605333,14.313871,14.473548,8.487667,1.853226,...,-0.235806,3.535000,8.724839,12.880333,16.234839,14.679677,10.001000,4.875484,-5.396333,-10.840645
Atlanta,9.745161,12.650714,13.417742,16.671000,19.840323,23.366000,26.612258,26.004516,22.471333,17.024516,...,14.256452,18.792333,20.569355,24.699333,27.168387,23.692903,23.371000,17.172258,12.946000,8.257097
Bellingham,5.737742,3.945357,6.397742,8.706000,15.197419,15.810667,18.750000,19.220645,15.275333,11.155806,...,7.756452,10.733333,13.235484,15.877667,18.184194,18.994839,16.331667,10.550000,8.056000,6.309355
Bend,0.164839,0.363571,1.016129,6.874667,13.843871,17.087000,22.243226,21.259032,14.659333,9.553226,...,4.888710,8.223667,12.697097,17.620667,22.061935,21.018710,17.795333,8.239032,6.519333,4.201290
Boise,1.043548,1.196786,3.605484,9.740333,18.396774,20.046667,27.855484,25.389032,19.784333,12.786129,...,8.474839,11.641667,17.453226,22.343667,26.432903,25.087742,21.710667,11.508387,8.097000,5.380968
Boston,2.648065,1.088571,4.380645,9.912000,15.218065,18.529333,24.115161,21.559677,18.980667,14.096452,...,5.335484,10.384333,14.830645,21.025333,24.739677,21.062903,18.084000,12.278710,5.475333,-1.176774
Bothell,5.824839,4.484643,6.458065,8.915000,16.247097,16.545000,19.974516,20.562903,15.941667,11.574839,...,8.145484,11.137000,13.804516,17.265667,19.910323,20.472581,17.521000,11.020968,8.743667,6.600968
Chicago,-0.541935,0.292857,2.273226,9.387333,13.612903,18.674333,21.687097,21.494194,18.377000,11.843871,...,5.497742,8.581667,12.221290,21.133667,23.588065,21.622581,18.830667,13.545484,5.207000,-3.074839


In [9]:
month_day = df['observed_date'].dt.strftime("%m-%d")

In [10]:
conditions = [
    (month_day >= '09-22') & (month_day < '12-21'),
    (month_day >= '12-21') | (month_day < '03-20'),
    (month_day >= '03-20') & (month_day < '06-21'),
    (month_day >= '06-21') & (month_day < '09-22')
]

seasons = ['fall', 'winter', 'spring', 'summer']

In [11]:
df['season'] = np.select(conditions, seasons, default='unknown')

In [12]:
df['season'].value_counts(dropna=False)

season
spring    8370
summer    8370
fall      8100
winter    8040
Name: count, dtype: int64

In [13]:
def year_season(row):
    return row['season'].title() + " " + str(row['observed_date'].year)

In [14]:
df['season_year'] = df.apply(year_season, axis=1)

In [15]:
seasonal_avg_temp_city = pd.pivot_table(data=df, columns='season_year', index='city', values='avg_temperature')

In [16]:
seasonal_avg_temp_city

season_year,Fall 2023,Fall 2024,Fall 2025,Spring 2023,Spring 2024,Spring 2025,Summer 2023,Summer 2024,Summer 2025,Winter 2023,Winter 2024,Winter 2025
city,,,,,,,,,,,,
Albuquerque,11.351444,11.627333,12.605778,17.005376,18.624516,18.707634,27.351505,26.162258,25.960968,4.389101,5.839556,5.837865
Anchorage,-1.539778,-1.546667,-1.347667,5.031720,5.594301,6.689785,13.245376,13.356452,14.454086,-6.871573,-7.813889,-5.063933
Atlanta,13.890444,14.943444,13.950222,18.921828,20.352903,20.103763,25.318817,25.514946,25.047527,11.066517,8.984111,8.789551
Bellingham,8.676889,8.707000,9.605778,12.140860,11.471290,12.512903,18.105269,17.685161,17.909247,5.433596,5.486111,3.787528
Bend,6.502778,6.445556,7.869333,10.581398,10.211075,11.624624,20.189032,20.536989,20.442366,0.627191,1.348111,0.702921
Boise,8.592222,8.470444,9.781444,14.058925,14.180215,15.885914,24.737957,25.569247,24.618817,1.484382,3.297667,2.397416
Boston,8.712556,8.754444,7.849222,12.708710,13.206989,13.191935,22.282581,22.446129,21.863441,2.411461,1.212333,-0.687303
Bothell,8.798889,9.105778,10.128000,12.682581,12.024516,13.199462,19.333656,19.219785,19.427527,5.520225,5.925444,4.332584
Chicago,8.043889,8.677667,7.205556,11.828710,13.683441,11.727419,20.840323,21.747849,22.004946,0.703371,1.161000,-1.609663


In [31]:
def get_season(season):
    if season.lower() not in seasons:
        return "Input valid Season"
    return seasonal_avg_temp_city.loc[:, seasonal_avg_temp_city.columns[seasonal_avg_temp_city.columns.str.contains(season.title())]]
    
def get_year(year):
    return seasonal_avg_temp_city.loc[:, seasonal_avg_temp_city.columns[seasonal_avg_temp_city.columns.str.contains(str(year))]]

In [33]:
get_year(2024).loc['Bothell', :]

season_year
Fall 2024       9.105778
Spring 2024    12.024516
Summer 2024    19.219785
Winter 2024     5.925444
Name: Bothell, dtype: float64